# Avaliação da Qualidade do Retrieval RAG

Este notebook compara os resultados da busca vetorial **antes** e **depois** do re-ranking para 3 queries de teste.

Fluxo: `Query → Busca Vetorial (top-20) → [SEM re-rank] vs [COM re-rank] → top-5`

In [5]:
import sys
import math
import re
from pathlib import Path
from typing import List

import chromadb
import pandas as pd

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

print('Imports OK')
print('Root:', ROOT)

Imports OK
Root: C:\Users\TG783NB\OneDrive - EY\Desktop\projeto_ia\development-program-ai_engineer\projects\project-1


In [6]:
DB_DIR = ROOT / 'chroma_db'
COLLECTION_NAME = 'compliance_knowledge'
INITIAL_K = 20
FINAL_K = 5


class SimpleEmbeddingFunction:
    def __call__(self, input):
        return [[float(len(text))] for text in input]

    def embed_documents(self, input):
        return [[float(len(text))] for text in input]

    def embed_query(self, input):
        if isinstance(input, str):
            return [[float(len(input))]]
        return [[float(len(text))] for text in input]

    def name(self):
        return 'simple'


embedding_fn = SimpleEmbeddingFunction()

client = chromadb.PersistentClient(path=str(DB_DIR))
collection = client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)
print('Total de chunks na base:', collection.count())

Total de chunks na base: 379


In [7]:
def tokenize(text):
    return re.findall(r'[a-zA-Z\u00C0-\u00FF0-9]+', text.lower())


def bm25_score(query_tokens, doc_tokens, k1=1.5, b=0.75, avg_dl=100.0):
    dl = len(doc_tokens)
    freq_map = {}
    for t in doc_tokens:
        freq_map[t] = freq_map.get(t, 0) + 1
    score = 0.0
    for term in query_tokens:
        if term not in freq_map:
            continue
        tf = freq_map[term]
        idf = math.log(1 + 1 / (0.5 + 0.5))
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * dl / avg_dl)
        score += idf * (numerator / denominator)
    return score


def coverage_bonus(query_tokens, doc_tokens):
    doc_set = set(doc_tokens)
    unique_query = set(query_tokens)
    if not unique_query:
        return 0.0
    return sum(1 for t in unique_query if t in doc_set) / len(unique_query)


def rerank_chunks(query, docs, metas, dists):
    query_tokens = tokenize(query)
    results = []
    for doc, meta, dist in zip(docs, metas, dists):
        doc_tokens = tokenize(doc)
        bm25 = bm25_score(query_tokens, doc_tokens)
        cov = coverage_bonus(query_tokens, doc_tokens)
        vec_sc = 1.0 / (1.0 + dist)
        rerank = 0.5 * bm25 + 0.3 * cov + 0.2 * vec_sc
        results.append({
            'source': meta.get('source', '?'),
            'chunk_id': meta.get('chunk_id', -1),
            'distance': round(dist, 4),
            'bm25': round(bm25, 4),
            'coverage': round(cov, 4),
            'vector_score': round(vec_sc, 4),
            'rerank_score': round(rerank, 4),
            'excerpt': doc[:200].replace('\n', ' '),
        })
    return results


print('Funcoes de re-ranking definidas')

Funcoes de re-ranking definidas


In [8]:
def evaluate_query(query, label=''):
    sep = '=' * 70
    print('\n' + sep)
    print('Query: ' + query)
    if label:
        print('(' + label + ')')
    print(sep)

    res = collection.query(
        query_texts=[query],
        n_results=min(INITIAL_K, collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )
    docs = res['documents'][0]
    metas = res['metadatas'][0]
    dists = res['distances'][0]

    before = [{
        'rank': i + 1,
        'source': m.get('source', '?'),
        'chunk_id': m.get('chunk_id', -1),
        'distance': round(d, 4),
        'excerpt': doc[:150].replace('\n', ' '),
    } for i, (doc, m, d) in enumerate(zip(docs, metas, dists))]

    reranked = sorted(
        rerank_chunks(query, docs, metas, dists),
        key=lambda x: x['rerank_score'],
        reverse=True
    )
    after = [{'rank': i + 1, **r} for i, r in enumerate(reranked)]

    df_before = pd.DataFrame(before[:FINAL_K])[['rank', 'source', 'chunk_id', 'distance', 'excerpt']]
    df_after = pd.DataFrame(after[:FINAL_K])[['rank', 'source', 'chunk_id', 'rerank_score', 'bm25', 'coverage', 'excerpt']]

    print('\nSEM RE-RANKING (ordem por distancia vetorial):')
    display(df_before)

    print('\nCOM RE-RANKING (ordem por score combinado):')
    display(df_after)

    return df_before, df_after


print('Funcao de avaliacao pronta')

Funcao de avaliacao pronta


## Query 1 — Suitability de investidor conservador

In [9]:
q1_before, q1_after = evaluate_query(
    query='Quais sao as regras de adequacao para investidores conservadores?',
    label='Suitability / perfil conservador'
)


Query: Quais sao as regras de adequacao para investidores conservadores?
(Suitability / perfil conservador)

SEM RE-RANKING (ordem por distancia vetorial):


,rank,source,chunk_id,distance,excerpt
0,1,politica_adequacao_investimento_v1.2.txt,5,196.0,**5. Apêndice** - Lista de produtos específico...
1,2,politica_adequacao_investimento_v1.2.txt,4,1521.0,- **4.3. Clientes Agressivos:** Podem receber ...
2,3,email_analise_cliente_01.txt,3,2025.0,Aguardamos sua análise de conformidade até o f...
3,4,analise_de_perfil_do_investidor.txt,5,2916.0,- Criptoativos (exceto via fundos com exposiçã...
4,5,politica_investimento_agressivo_v1.0.txt,3,5184.0,- **Ações de uma única empresa:** A concentraç...



COM RE-RANKING (ordem por score combinado):


,rank,source,chunk_id,rerank_score,bm25,coverage,excerpt
0,1,politica_investimento_agressivo_v1.0.txt,0,1.8921,3.5842,0.3333,# Política de Investimento para Perfil Agressi...
1,2,politica_adequacao_investimento_v1.2.txt,0,1.8604,3.5208,0.3333,**Política de Adequação de Perfil de Investime...
2,3,resol_030_cvm.pdf,28,1.3624,2.5914,0.2222,do consultor por ele contratado. CAPÍTULO VII...
3,4,email_analise_cliente_01.txt,0,1.3044,2.4755,0.2222,**DE:** compliance.interna@fso-ey.com **PARA:*...
4,5,resol_030_cvm.pdf,1,1.2120,2.2906,0.2222,"www.cvm.gov.br RESOLUÇÃO CVM Nº 30, DE 11 DE ..."


## Query 2 — Recomendacao de criptomoedas para perfil conservador

In [10]:
q2_before, q2_after = evaluate_query(
    query='Posso recomendar criptomoedas e ativos de alto risco para clientes conservadores?',
    label='Conformidade de recomendacao - risco elevado'
)


Query: Posso recomendar criptomoedas e ativos de alto risco para clientes conservadores?
(Conformidade de recomendacao - risco elevado)

SEM RE-RANKING (ordem por distancia vetorial):


,rank,source,chunk_id,distance,excerpt
0,1,politica_adequacao_investimento_v1.2.txt,5,4.0,**5. Apêndice** - Lista de produtos específico...
1,2,politica_adequacao_investimento_v1.2.txt,4,529.0,- **4.3. Clientes Agressivos:** Podem receber ...
2,3,email_analise_cliente_01.txt,3,841.0,Aguardamos sua análise de conformidade até o f...
3,4,analise_de_perfil_do_investidor.txt,5,1444.0,- Criptoativos (exceto via fundos com exposiçã...
4,5,politica_investimento_agressivo_v1.0.txt,3,3136.0,- **Ações de uma única empresa:** A concentraç...



COM RE-RANKING (ordem por score combinado):


,rank,source,chunk_id,rerank_score,bm25,coverage,excerpt
0,1,politica_investimento_agressivo_v1.0.txt,0,2.9854,5.6981,0.4545,# Política de Investimento para Perfil Agressi...
1,2,manual_comunicacao_cliente_v1.0.txt,2,2.6473,5.0764,0.3636,**3. Recomendações Obrigatórias** - **3.1. Ale...
2,3,politica_adequacao_investimento_v1.2.txt,2,2.4709,4.7236,0.3636,**3. Classificação de Risco dos Produtos** - *...
3,4,analise_de_perfil_do_investidor.txt,5,2.4441,4.6696,0.3636,- Criptoativos (exceto via fundos com exposiçã...
4,5,politica_adequacao_investimento_v1.2.txt,5,1.9715,3.6994,0.2727,**5. Apêndice** - Lista de produtos específico...


## Query 3 — Comunicacao e divulgacao de produtos de investimento

In [11]:
q3_before, q3_after = evaluate_query(
    query='Quais sao as obrigacoes de comunicacao na distribuicao de produtos de investimento pela ANBIMA?',
    label='Compliance de comunicacao - ANBIMA'
)


Query: Quais sao as obrigacoes de comunicacao na distribuicao de produtos de investimento pela ANBIMA?
(Compliance de comunicacao - ANBIMA)

SEM RE-RANKING (ordem por distancia vetorial):


,rank,source,chunk_id,distance,excerpt
0,1,politica_adequacao_investimento_v1.2.txt,4,81.0,- **4.3. Clientes Agressivos:** Podem receber ...
1,2,email_analise_cliente_01.txt,3,225.0,Aguardamos sua análise de conformidade até o f...
2,3,politica_adequacao_investimento_v1.2.txt,5,256.0,**5. Apêndice** - Lista de produtos específico...
3,4,analise_de_perfil_do_investidor.txt,5,576.0,- Criptoativos (exceto via fundos com exposiçã...
4,5,politica_investimento_agressivo_v1.0.txt,3,1764.0,- **Ações de uma única empresa:** A concentraç...



COM RE-RANKING (ordem por score combinado):


,rank,source,chunk_id,rerank_score,bm25,coverage,excerpt
0,1,politica_adequacao_investimento_v1.2.txt,0,4.0477,7.8953,0.3333,**Política de Adequação de Perfil de Investime...
1,2,politica_investimento_agressivo_v1.0.txt,0,3.3375,6.5250,0.2500,# Política de Investimento para Perfil Agressi...
2,3,resol_030_cvm.pdf,1,3.1184,6.0868,0.2500,"www.cvm.gov.br RESOLUÇÃO CVM Nº 30, DE 11 DE ..."
3,4,politica_adequacao_investimento_v1.2.txt,4,2.8142,5.5234,0.1667,- **4.3. Clientes Agressivos:** Podem receber ...
4,5,manual_comunicacao_cliente_v1.0.txt,2,2.7779,5.4557,0.1667,**3. Recomendações Obrigatórias** - **3.1. Ale...


## Analise Comparativa Final

In [12]:
def position_changes(before_df, after_df):
    before_ids = list(zip(before_df['source'], before_df['chunk_id']))
    after_ids = list(zip(after_df['source'], after_df['chunk_id']))
    changed = sum(1 for b, a in zip(before_ids, after_ids) if b != a)
    new_entries = len(set(after_ids) - set(before_ids))
    return {'posicoes_alteradas': changed, 'novos_chunks_promovidos': new_entries}


summary = pd.DataFrame([
    {'query': 'Q1 - Perfil conservador', **position_changes(q1_before, q1_after)},
    {'query': 'Q2 - Criptomoedas / alto risco', **position_changes(q2_before, q2_after)},
    {'query': 'Q3 - Comunicacao ANBIMA', **position_changes(q3_before, q3_after)},
])

print('Impacto do Re-ranking:')
display(summary)

Impacto do Re-ranking:


,query,posicoes_alteradas,novos_chunks_promovidos
0,Q1 - Perfil conservador,5,5
1,Q2 - Criptomoedas / alto risco,4,3
2,Q3 - Comunicacao ANBIMA,5,4
